# 智慧垃圾分類 — YOLO26m 訓練 Notebook（5 類版，v4）

**Branch: feature/yolo26 ｜ 執行前必做：Runtime → Change runtime type → T4 GPU**

## 執行順序（v4：5 類，TrashNet only）
1. Cell 1：確認 GPU
2. Cell 2：安裝套件
3. Cell 3：Clone 專案（feature/yolo26）
4. Cell 4：下載 TrashNet
5. Cell 5：TrashNet 格式轉換（6 類格式）
6. **Cell 5b：重標為 5 類（移除鋁箔包，Class 5→4）**
7. ~~Cell 6 / 6b / 6c：TACO（**跳過**，TACO 標籤雜訊問題）~~
8. **Cell 7：訓練 yolo26m，waste_sorter_yolo26_v4**
9. **Cell 8：下載 best_v4.pt**

> **決策紀錄**：TACO 資料集加入後 mAP 從 0.795 跌至 0.439，判斷一般垃圾標籤過於雜亂。
> 捨棄鋁箔包類別，改為 5 類系統（Class 4 由組員補充後再恢復）。

In [ ]:
# ── Cell 1：確認 GPU ───────────────────────────────────────────────────────────
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('❌ 沒有 GPU，請先到 Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2：安裝套件（只升級 ultralytics，不動 Pillow 避免衝突）──────────────
!pip install -U ultralytics -q          # 升到最新，確保 YOLO26 支援
!pip install kaggle requests -q         # 不升 Pillow，避免與 Colab 內建版衝突
import ultralytics
print(f'ultralytics 版本：{ultralytics.__version__}')
print('套件安裝完成')

In [ ]:
# ── Cell 3：Clone 專案（feature/yolo26 分支）─────────────────────────────────
import os

REPO = 'AI-course'
if os.path.exists(REPO):
    print('Repo 已存在，執行 git pull...')
    !cd {REPO} && git fetch origin && git checkout feature/yolo26 && git pull
else:
    !git clone -b feature/yolo26 https://github.com/Saibusu/AI-course.git

%cd /content/AI-course
print('工作目錄：', os.getcwd())
print('目前分支：', os.popen('git branch --show-current').read().strip())

In [ ]:
# ── Cell 4：下載 TrashNet（Kaggle API Token）───────────────────────────────────
# 取得方式：kaggle.com/settings → API → 複製 Token（格式：KGAT_...）
import os

os.environ['KAGGLE_TOKEN'] = 'YOUR_KAGGLE_API_TOKEN'  # ← 貼上你的 Token，勿上傳 GitHub

!kaggle datasets download -d asdasdasasdas/garbage-classification -p data/
!unzip -q data/garbage-classification.zip -d data/TrashNet

# 確認結構
for root, dirs, files in os.walk('data/TrashNet'):
    level = root.replace('data/TrashNet', '').count(os.sep)
    if level == 2:
        print(f'{os.path.basename(root)}/: {len(files)} files')
    if level > 2:
        break

In [ ]:
# ── Cell 5：TrashNet → YOLO 6-class 格式轉換 ──────────────────────────────────
# 已知：Kaggle 資料集實際路徑為兩層子目錄
# data/TrashNet/garbage classification/Garbage classification/{glass,metal,...}/
import os

TRASHNET_DIR = 'data/TrashNet/garbage classification/Garbage classification'

if not os.path.exists(TRASHNET_DIR):
    # 嘗試大寫版本
    TRASHNET_DIR = 'data/TrashNet/Garbage classification/Garbage classification'

print('TrashNet 來源路徑:', TRASHNET_DIR)
print('子目錄：', os.listdir(TRASHNET_DIR))

!python data/prepare_trashnet.py \
    --trashnet-dir "{TRASHNET_DIR}" \
    --output-dir   data/trashnet_yolo

for split in ['train', 'val', 'test']:
    imgs = len(list(os.scandir(f'data/trashnet_yolo/{split}/images')))
    print(f'  {split}: {imgs} images')

In [ ]:
# ── Cell 5b：重標為 5 類（移除鋁箔包 Class 4，一般垃圾 Class 5→4）────────────
# 輸入：data/trashnet_yolo（6-class，Class 4 = 鋁箔包，Class 5 = 一般垃圾）
# 輸出：data/trashnet_5class（5-class，Class 4 = 一般垃圾）
import os, glob, shutil

SRC = 'data/trashnet_yolo'
DST = 'data/trashnet_5class'

for split in ['train', 'val', 'test']:
    os.makedirs(f'{DST}/{split}/images', exist_ok=True)
    os.makedirs(f'{DST}/{split}/labels', exist_ok=True)

    # 重標 labels
    for lbl_path in glob.glob(f'{SRC}/{split}/labels/*.txt'):
        new_lines = []
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                cls = int(parts[0])
                if cls == 4:
                    continue       # 鋁箔包 → 刪除
                if cls == 5:
                    cls = 4        # 一般垃圾 5→4
                new_lines.append(f"{cls} {' '.join(parts[1:])}\n")
        new_lbl = lbl_path.replace(SRC, DST)
        with open(new_lbl, 'w') as f:
            f.writelines(new_lines)

    # 複製圖片
    for img_path in glob.glob(f'{SRC}/{split}/images/*'):
        dst_img = img_path.replace(SRC, DST)
        if not os.path.exists(dst_img):
            shutil.copy2(img_path, dst_img)

# 寫入 5-class data.yaml
yaml_content = f"""path: /content/AI-course/data/trashnet_5class
train: train/images
val:   val/images
test:  test/images

nc: 5
names:
  0: 寶特瓶
  1: 鐵鋁罐
  2: 紙餐盒
  3: 塑膠袋
  4: 一般垃圾
"""
with open(f'{DST}/data.yaml', 'w', encoding='utf-8') as f:
    f.write(yaml_content)

print('5-class 資料集建立完成（鋁箔包已移除）')
for split in ['train', 'val', 'test']:
    n = len(glob.glob(f'{DST}/{split}/images/*'))
    print(f'  {split}: {n} images')

In [ ]:
# ── Cell 6：下載 TACO + 合併三層資料集 ────────────────────────────────────────
# ADR-002 Layer 1（主力）：含鋁箔包（Drink carton）標註
# 預計下載時間：20–40 分鐘（Flickr 圖片）
import os

# Step 1：Clone TACO repo（若已存在則跳過）
if not os.path.exists('data/TACO_repo'):
    !git clone https://github.com/pedropro/TACO.git data/TACO_repo
else:
    print('TACO_repo 已存在，跳過 clone')

# Step 2：確認 annotations.json 存在
ann_path = 'data/TACO_repo/data/annotations.json'
print('annotations.json 存在：', os.path.exists(ann_path))
print('TACO data/ 內容：', os.listdir('data/TACO_repo/data') if os.path.exists('data/TACO_repo/data') else 'NOT FOUND')

In [ ]:
# ── Cell 6b：從 Zenodo 下載完整 TACO（官方備份，不依賴 Flickr）──────────────
# Zenodo DOI: 10.5281/zenodo.3587843  共 2.7GB，下載約 5–10 分鐘
import os, glob

TACO_ZIP = 'data/TACO.zip'
TACO_OUT = 'data/TACO_zenodo'

if not os.path.exists(TACO_OUT):
    print('下載 TACO 完整資料集（2.7GB）...')
    !wget --show-progress "https://zenodo.org/record/3587843/files/TACO.zip" -O {TACO_ZIP}
    print('解壓中...')
    !unzip -q {TACO_ZIP} -d {TACO_OUT}
    print('解壓完成')
else:
    print('TACO_zenodo 已存在，跳過下載')

# 確認結構
ann_files = glob.glob(f'{TACO_OUT}/**/annotations.json', recursive=True)
print('annotations.json 位置：', ann_files)
img_count = len(glob.glob(f'{TACO_OUT}/**/*.jpg', recursive=True))
print(f'圖片總數：{img_count}')

In [ ]:
# ── Cell 7：訓練 yolo26m，5 類，waste_sorter_yolo26_v4 ───────────────────────
# 策略：移除鋁箔包，用乾淨的 TrashNet 5-class 資料重訓
import os
from ultralytics import YOLO

DATA_YAML = 'data/trashnet_5class/data.yaml'
print('使用 trashnet_5class（5 類，無鋁箔包，無 TACO 雜訊）')

model = YOLO('yolo26m.pt')
print('模型：yolo26m.pt（YOLO26 medium，ADR-001 ✅）')

results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,
    project='runs/train',
    name='waste_sorter_yolo26_v4',
    exist_ok=True,
    patience=20,
    cos_lr=True,
    lr0=1e-3,
    lrf=1e-2,
    mosaic=1.0,
    fliplr=0.5,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    hsv_s=0.7,
    hsv_v=0.4,
    mixup=0.1,
)

mAP = results.results_dict.get('metrics/mAP50(B)', 'N/A')
print(f'\n訓練完成！mAP@50 = {mAP}')
print('模型路徑：runs/train/waste_sorter_yolo26_v4/weights/best.pt')

In [ ]:
# ── Cell 8：下載 best_v4.pt ───────────────────────────────────────────────────
import glob, shutil, os
from google.colab import files

pts = glob.glob('**/best.pt', recursive=True)
print('找到的 best.pt：', pts)

src = next((p for p in pts if 'waste_sorter_yolo26_v4' in p), None)
if src is None:
    src = next((p for p in pts if 'waste_sorter_yolo26_v2' in p), None)
if src is None:
    src = next(p for p in pts if p != 'best_v4.pt')

print(f'使用模型：{src}')
shutil.copy(src, 'best_v4.pt')
print(f'Model size: {os.path.getsize("best_v4.pt")/1e6:.1f} MB')

files.download('best_v4.pt')
print('\n✅ 下載完成。接著在筆電執行：')
print('scp best_v4.pt jetson@<JETSON_IP>:~/AI-course/models/best.pt')

In [ ]:
# ── Cell 8：下載 best_v3.pt ───────────────────────────────────────────────────
import glob, shutil, os
from google.colab import files

pts = glob.glob('**/best.pt', recursive=True)
print('找到的 best.pt：', pts)

# 優先取 v2（yolo26m），再找 yolo26，避免 SameFileError
src = next((p for p in pts if 'waste_sorter_yolo26_v2' in p), None)
if src is None:
    src = next((p for p in pts if 'waste_sorter_yolo26' in p), None)
if src is None:
    src = next(p for p in pts if p != 'best_v3.pt')

print(f'使用模型：{src}')
shutil.copy(src, 'best_v3.pt')
print(f'Model size: {os.path.getsize("best_v3.pt")/1e6:.1f} MB')

files.download('best_v3.pt')
print('\n✅ 下載完成。接著在筆電執行：')
print('scp best_v3.pt jetson@<JETSON_IP>:~/AI-course/models/best.pt')